# Script 12 — Literature-Grounded Volatility GARCH/EGARCH/GJR Ensemble (Corrected)

This corrected version preserves the original modelling objective and output filenames while adding safeguards for unstable fits, correcting EWMA timing, and making model selection use common forecast origins.

## Corrections in this version

- Reject non-converged or implausible GARCH-family parameter estimates; log every rejected fit/forecast.
- Use the return observed at forecast origin \(t\) in the EWMA variance update, consistent with forecasting returns after \(t\).
- Reject numerically explosive raw variance forecasts rather than silently clipping them. The threshold is a transparent numerical-stability guardrail, and every rejection is logged.
- Rank eligible candidates on a common set of inner-year forecast origins; candidates with less than 90% coverage are not eligible for selection.
- Apply an explicit EWMA fallback when a selected component forecast is unavailable in an outer year, preserving a transparent full-coverage operational forecast.
- Keep the development period restricted to dates before 2024. Do not load or evaluate the 2024–2025 holdout.

The numerical-stability threshold is not a claim that all larger forecasts are impossible; it is a conservative screening rule to identify model outputs that should not be accepted without investigation. Review the generated audit files before interpreting performance.


In [ ]:
from pathlib import Path
import warnings, time
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t as student_t
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

CURRENCIES=["USD","GBP","EUR","JPY"]
HORIZONS=[1,3,7]
ALL_FORECAST_YEARS=[2018,2019,2020,2021,2022,2023]
OUTER_YEARS=[2019,2020,2021,2022,2023]
REFIT_EVERY=40
MIN_TRAIN=500
EPS=1e-10

MODEL_SPECS={
    "GARCH_NORMAL":dict(vol="GARCH",p=1,o=0,q=1,dist="normal"),
    "GARCH_T":dict(vol="GARCH",p=1,o=0,q=1,dist="t"),
    "EGARCH_T":dict(vol="EGARCH",p=1,o=1,q=1,dist="t"),
    "GJR_GARCH_T":dict(vol="GARCH",p=1,o=1,q=1,dist="t"),
}
BASELINE_MODELS=["EWMA94","HIST20","HIST60"]
print("Parametric models:", list(MODEL_SPECS))

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        p=root/"data"/"processed"/"02_development_model_dataset.csv"
        if p.exists():
            return root.resolve()
    raise FileNotFoundError("Could not find data/processed/02_development_model_dataset.csv")

ROOT=find_root()
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
PRED_DIR=ROOT/"outputs"/"predictions"
TABLE_DIR=ROOT/"outputs"/"tables"
PRED_DIR.mkdir(parents=True,exist_ok=True)
TABLE_DIR.mkdir(parents=True,exist_ok=True)

df=pd.read_csv(DEV_PATH,parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
assert df["Date"].max() < pd.Timestamp("2024-01-01"), "STOP: final holdout entered Script 12"
print("Development range:",df["Date"].min().date(),"to",df["Date"].max().date())
print("PASS — 2024–2025 remains untouched")

In [ ]:
series={}
for cur in CURRENCIES:
    mid=pd.to_numeric(df[f"{cur}_mid"],errors="coerce")
    ret=100*np.log(mid/mid.shift(1))
    temp=pd.DataFrame({"Date":df["Date"],"Mid":mid,"Return":ret})
    sq=temp["Return"]**2
    for h in HORIZONS:
        rv=pd.Series(0.0,index=temp.index)
        for j in range(1,h+1):
            rv=rv+sq.shift(-j)
        temp[f"RV_H{h}"]=rv
    series[cur]=temp
print(series["USD"].head(10))

## Diagnostics

ADF checks return stationarity. ARCH-LM checks conditional heteroskedasticity. Jarque–Bera and kurtosis document non-normal/heavy-tailed behaviour. These diagnostics justify rather than select the GARCH family.

In [ ]:
diag=[]
for cur in CURRENCIES:
    r=series[cur]["Return"].dropna()
    adf_stat,adf_p,*_=adfuller(r,autolag="AIC")
    arch_stat,arch_p,*_=het_arch(r,nlags=12)
    jb=stats.jarque_bera(r)
    kurt=stats.kurtosis(r,fisher=False,bias=False)
    diag.append({
        "Currency":cur,
        "Observations":len(r),
        "ADF_pvalue":adf_p,
        "ARCH_LM_pvalue":arch_p,
        "Jarque_Bera_pvalue":jb.pvalue,
        "Pearson_Kurtosis":kurt,
        "ARCH_Effects_5pct":arch_p<0.05,
        "Heavy_Tails_Kurtosis_gt3":kurt>3,
    })
diagnostics=pd.DataFrame(diag)
display(diagnostics)
diagnostics.to_csv(TABLE_DIR/"12_volatility_diagnostics.csv",index=False)

In [ ]:
def qlike(rv,vf):
    rv=np.asarray(rv,float); vf=np.asarray(vf,float)
    m=np.isfinite(rv)&np.isfinite(vf)&(vf>EPS)
    if not m.any(): return np.nan
    rv=np.maximum(rv[m],EPS); vf=np.maximum(vf[m],EPS)
    return float(np.mean(np.log(vf)+rv/vf))

def qlike_losses(rv,vf):
    rv=np.maximum(np.asarray(rv,float),EPS)
    vf=np.maximum(np.asarray(vf,float),EPS)
    return np.log(vf)+rv/vf

def variance_mse(rv,vf):
    rv=np.asarray(rv,float); vf=np.asarray(vf,float)
    m=np.isfinite(rv)&np.isfinite(vf)
    return float(np.mean((rv[m]-vf[m])**2)) if m.any() else np.nan

def vol_rmse(rv,vf):
    rv=np.asarray(rv,float); vf=np.asarray(vf,float)
    m=np.isfinite(rv)&np.isfinite(vf)
    if not m.any(): return np.nan
    return float(np.sqrt(np.mean((np.sqrt(np.maximum(rv[m],0))-np.sqrt(np.maximum(vf[m],0)))**2)))

def dm_loss_test(rv,model_v,benchmark_v,horizon=1):
    rv=np.asarray(rv,float); model_v=np.asarray(model_v,float); benchmark_v=np.asarray(benchmark_v,float)
    m=np.isfinite(rv)&np.isfinite(model_v)&np.isfinite(benchmark_v)&(model_v>0)&(benchmark_v>0)
    rv=rv[m]; model_v=model_v[m]; benchmark_v=benchmark_v[m]
    if len(rv)<30: return np.nan,np.nan
    d=qlike_losses(rv,benchmark_v)-qlike_losses(rv,model_v) # positive = model better
    n=len(d); lag=max(0,horizon-1)
    lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if n<=k: break
        cov=np.cov(d[k:],d[:-k],ddof=1)[0,1]
        lrv += 2*(1-k/(lag+1))*cov
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n)
    p=2*student_t.sf(abs(stat),df=n-1)
    return float(stat),float(p)

In [ ]:
# Baselines: EWMA must include the return observed at the forecast origin.
MAX_RAW_TO_EWMA_RATIO = 1000.0
MIN_MODEL_COVERAGE = 0.90
MIN_COMMON_ORIGIN_COVERAGE = 0.80

def ewma_daily_variance(return_series, lam=0.94):
    """Daily EWMA variance available after observing the return at each date."""
    r2 = pd.to_numeric(return_series, errors="coerce").pow(2).to_numpy(dtype=float)
    out = np.full(len(r2), np.nan, dtype=float)
    for i in range(len(r2)):
        if not np.isfinite(r2[i]):
            continue
        if i == 0 or not np.isfinite(out[i-1]):
            history = r2[:i+1]
            history = history[np.isfinite(history)]
            out[i] = float(np.mean(history)) if len(history) else np.nan
        else:
            # Uses r_t observed at the forecast origin t (not r_{t-1}).
            out[i] = lam * out[i-1] + (1-lam) * r2[i]
    return pd.Series(out, index=return_series.index, dtype=float)

for cur in CURRENCIES:
    series[cur]["EWMA_Daily_Variance"] = ewma_daily_variance(series[cur]["Return"])

def add_baselines(temp, h):
    out = temp.copy()
    r2 = out["Return"].pow(2)
    # Historical variance uses returns available through the current origin.
    hist20 = r2.rolling(20, min_periods=20).mean()
    hist60 = r2.rolling(60, min_periods=60).mean()
    ewma = out["EWMA_Daily_Variance"]
    out[f"EWMA94_H{h}"] = h * ewma
    out[f"HIST20_H{h}"] = h * hist20
    out[f"HIST60_H{h}"] = h * hist60
    return out


## Rolling parametric forecasts

Parameters are re-estimated every **40 trading observations**. Between re-estimation points the same estimated parameters are applied to the newly available return history, so the conditional variance updates as new returns arrive without re-tuning the model every day.

For EGARCH multi-step forecasts, simulation is used because multi-step analytic forecasts are not generally available for EGARCH.

In [ ]:
# Robust rolling parametric forecasts with explicit fit/forecast audit logging.
forecast_audit_rows = []

def record_forecast_issue(cur, h, year, origin, model_name, stage, reason, details=""):
    forecast_audit_rows.append({
        "Currency": cur, "Horizon": h, "Year": year,
        "Forecast_Origin": pd.Timestamp(origin) if pd.notna(origin) else pd.NaT,
        "Model": model_name, "Stage": stage, "Reason": reason,
        "Details": str(details)[:1000]
    })

def validate_fit(model_name, fit):
    """Return (valid, reason) for convergence, finiteness, and basic parameter stability."""
    try:
        if int(getattr(fit, "convergence_flag", 1)) != 0:
            return False, f"optimizer_convergence_flag={getattr(fit, 'convergence_flag', 'missing')}"
        p = fit.params
        if not np.isfinite(np.asarray(p, dtype=float)).all():
            return False, "non_finite_parameter"
        get = lambda name: float(p[name]) if name in p.index else np.nan
        omega = get("omega")
        if not np.isfinite(omega) or omega <= 0:
            return False, "invalid_omega"
        if model_name in ("GARCH_NORMAL", "GARCH_T"):
            alpha, beta = get("alpha[1]"), get("beta[1]")
            if not np.isfinite([alpha, beta]).all() or alpha < 0 or beta < 0:
                return False, "invalid_garch_coefficients"
            if alpha + beta >= 0.9995:
                return False, "garch_persistence_at_or_above_0.9995"
        elif model_name == "GJR_GARCH_T":
            alpha, gamma, beta = get("alpha[1]"), get("gamma[1]"), get("beta[1]")
            if not np.isfinite([alpha, gamma, beta]).all() or alpha < 0 or beta < 0 or alpha + gamma < 0:
                return False, "invalid_gjr_coefficients"
            if alpha + beta + 0.5 * gamma >= 0.9995:
                return False, "gjr_persistence_at_or_above_0.9995"
        elif model_name == "EGARCH_T":
            beta = get("beta[1]")
            if not np.isfinite(beta) or abs(beta) >= 0.9995:
                return False, "egarch_beta_abs_at_or_above_0.9995"
        if model_name.endswith("_T"):
            nu = get("nu")
            if not np.isfinite(nu) or nu <= 2.05:
                return False, "student_t_degrees_of_freedom_at_or_below_2.05"
        return True, "valid"
    except Exception as e:
        return False, f"parameter_validation_error:{type(e).__name__}:{e}"

def forecast_model_year(cur, h, year, model_name):
    spec = MODEL_SPECS[model_name]
    temp = series[cur].copy()
    idxs = temp.index[
        temp["Date"].dt.year.eq(year)
        & temp[f"RV_H{h}"].notna()
        & temp["Return"].notna()
    ].tolist()
    if not idxs:
        return pd.DataFrame()

    rows = []
    params = None
    retry_after_pos = 0

    for pos, idx in enumerate(idxs):
        origin = temp.loc[idx, "Date"]
        hist = temp.loc[:idx, "Return"].dropna()
        if len(hist) < MIN_TRAIN:
            record_forecast_issue(cur, h, year, origin, model_name, "fit",
                                  "insufficient_training_history", f"n={len(hist)}")
            continue

        should_refit = params is None or (pos % REFIT_EVERY == 0)
        if params is None and pos < retry_after_pos:
            continue

        if should_refit:
            try:
                am = arch_model(hist, mean="Constant", rescale=False, **spec)
                fit = am.fit(disp="off", show_warning=False, options={"maxiter": 500})
                ok, reason = validate_fit(model_name, fit)
                if not ok:
                    record_forecast_issue(cur, h, year, origin, model_name, "fit",
                                          "rejected_fit", reason)
                    params = None
                    retry_after_pos = pos + 5
                    continue
                params = fit.params.to_numpy(dtype=float)
            except Exception as e:
                record_forecast_issue(cur, h, year, origin, model_name, "fit",
                                      "fit_exception", f"{type(e).__name__}: {e}")
                params = None
                retry_after_pos = pos + 5
                continue

        try:
            am_now = arch_model(hist, mean="Constant", rescale=False, **spec)
            fixed = am_now.fix(params)
            if model_name == "EGARCH_T" and h > 1:
                fc = fixed.forecast(
                    horizon=h, reindex=False, method="simulation", simulations=1000
                )
            else:
                fc = fixed.forecast(horizon=h, reindex=False, method="analytic")

            path = np.asarray(fc.variance.iloc[-1, :h], dtype=float)
            v = float(np.sum(path))
            if not np.isfinite(path).all() or not np.isfinite(v) or v <= EPS:
                record_forecast_issue(cur, h, year, origin, model_name, "forecast",
                                      "non_finite_or_non_positive_forecast", repr(path.tolist()))
                params = None
                retry_after_pos = pos + 1
                continue

            anchor_daily = float(temp.loc[idx, "EWMA_Daily_Variance"])
            anchor_h = h * anchor_daily if np.isfinite(anchor_daily) and anchor_daily > 0 else np.nan
            ratio = v / anchor_h if np.isfinite(anchor_h) and anchor_h > EPS else np.nan
            if np.isfinite(ratio) and ratio > MAX_RAW_TO_EWMA_RATIO:
                record_forecast_issue(
                    cur, h, year, origin, model_name, "forecast",
                    "raw_forecast_exceeds_numerical_guardrail",
                    f"forecast={v:.8g}; EWMA_anchor={anchor_h:.8g}; ratio={ratio:.8g}; "
                    f"threshold={MAX_RAW_TO_EWMA_RATIO}"
                )
                # Do not clip an invalid-looking forecast; force a fresh fit at the next origin.
                params = None
                retry_after_pos = pos + 1
                continue

            rows.append({
                "Currency": cur, "Horizon": h, "Year": year,
                "Forecast_Origin": pd.Timestamp(origin),
                "Realized_Variance": float(temp.loc[idx, f"RV_H{h}"]),
                "Model": model_name,
                "Variance_Forecast": max(v, EPS),
                "Raw_Forecast_to_EWMA_Ratio": ratio
            })
        except Exception as e:
            record_forecast_issue(cur, h, year, origin, model_name, "forecast",
                                  "forecast_exception", f"{type(e).__name__}: {e}")
            params = None
            retry_after_pos = pos + 1
            continue

    return pd.DataFrame(rows)


In [ ]:
parametric_parts=[]
for cur in CURRENCIES:
    for h in HORIZONS:
        for year in ALL_FORECAST_YEARS:
            print(f"\n===== {cur} H{h} {year} =====",flush=True)
            for model_name in MODEL_SPECS:
                t0=time.time()
                part=forecast_model_year(cur,h,year,model_name)
                if not part.empty: parametric_parts.append(part)
                print(model_name,"rows=",len(part),"sec=",round(time.time()-t0,1),flush=True)

if not parametric_parts:
    raise RuntimeError("No parametric volatility forecasts were generated. Check the first FIT FAIL message.")
parametric_long=pd.concat(parametric_parts,ignore_index=True)
print("Parametric forecasts:",len(parametric_long))

In [ ]:
# Build candidate file from valid forecasts only. Rejected/failed forecasts are retained in the audit.
baseline_parts = []
for cur in CURRENCIES:
    for h in HORIZONS:
        temp = add_baselines(series[cur], h)
        use = temp[
            temp["Date"].dt.year.isin(ALL_FORECAST_YEARS)
            & temp[f"RV_H{h}"].notna()
            & temp["Return"].notna()
        ].copy()
        for name in BASELINE_MODELS:
            baseline_parts.append(pd.DataFrame({
                "Currency": cur,
                "Horizon": h,
                "Year": use["Date"].dt.year,
                "Forecast_Origin": use["Date"],
                "Realized_Variance": use[f"RV_H{h}"],
                "Model": name,
                "Variance_Forecast": use[f"{name}_H{h}"],
                "Raw_Forecast_to_EWMA_Ratio": np.where(
                    use[f"EWMA94_H{h}"].to_numpy(dtype=float) > EPS,
                    use[f"{name}_H{h}"].to_numpy(dtype=float)
                    / use[f"EWMA94_H{h}"].to_numpy(dtype=float),
                    np.nan
                )
            }))

baseline_long = pd.concat(baseline_parts, ignore_index=True)
candidate_long = pd.concat([parametric_long, baseline_long], ignore_index=True)
candidate_long["Forecast_Origin"] = pd.to_datetime(candidate_long["Forecast_Origin"])
candidate_long = candidate_long[
    np.isfinite(candidate_long["Variance_Forecast"])
    & (candidate_long["Variance_Forecast"] > 0)
    & np.isfinite(candidate_long["Realized_Variance"])
].copy()

candidate_path = PRED_DIR / "12_volatility_candidate_predictions_2018_2023.csv"
candidate_long.to_csv(candidate_path, index=False)

# Coverage denominator is every valid forecast origin for the target/horizon/year.
coverage_rows = []
for cur in CURRENCIES:
    temp = series[cur]
    for h in HORIZONS:
        for year in ALL_FORECAST_YEARS:
            expected = temp.loc[
                temp["Date"].dt.year.eq(year)
                & temp[f"RV_H{h}"].notna()
                & temp["Return"].notna(), "Date"
            ]
            denom = int(expected.nunique())
            sub = candidate_long[
                (candidate_long["Currency"] == cur)
                & (candidate_long["Horizon"] == h)
                & (candidate_long["Year"] == year)
            ]
            for model_name in sorted(set(BASELINE_MODELS) | set(MODEL_SPECS)):
                got = int(sub.loc[sub["Model"].eq(model_name), "Forecast_Origin"].nunique())
                coverage_rows.append({
                    "Currency": cur, "Horizon": h, "Year": year,
                    "Model": model_name, "Expected_Origins": denom,
                    "Available_Origins": got,
                    "Coverage_Rate": got / denom if denom else np.nan,
                    "Missing_Origins": max(0, denom-got)
                })
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(TABLE_DIR / "12_volatility_candidate_coverage.csv", index=False)

audit_df = pd.DataFrame(forecast_audit_rows)
audit_df.to_csv(TABLE_DIR / "12_volatility_forecast_audit.csv", index=False)

print("Candidate forecasts:", len(candidate_long))
print("Forecast issues logged:", len(audit_df))
print("Candidate predictions:", candidate_path)
print("Audit:", TABLE_DIR / "12_volatility_forecast_audit.csv")
print("Coverage:", TABLE_DIR / "12_volatility_candidate_coverage.csv")
display(coverage_df.query("Currency == 'USD' and Horizon in [3, 7]").head(30))


## Forecast amalgamation with matched-origin safeguards

For each outer year Y, candidate models are evaluated using **Y−1 only**. Before ranking, a model must have at least 90% of expected origins in the inner year. Eligible candidates are scored on the same common set of origins, and the selected strategy is frozen for year Y.

If a selected component is unavailable in the outer year, the operational forecast uses the corrected EWMA baseline for that origin. This fallback is recorded explicitly, so it cannot silently hide model failures.


In [ ]:
# Forecast amalgamation with eligibility and matched-origin rules.
def expected_target_frame(cur, h, year):
    temp = series[cur]
    mask = (
        temp["Date"].dt.year.eq(year)
        & temp[f"RV_H{h}"].notna()
        & temp["Return"].notna()
    )
    return temp.loc[mask, ["Date", f"RV_H{h}"]].rename(
        columns={"Date": "Forecast_Origin", f"RV_H{h}": "Realized_Variance"}
    ).copy()

def model_year_table(cur, h, year):
    """Rank eligible models on the same common origins in the inner year."""
    g = candidate_long[
        (candidate_long["Currency"] == cur)
        & (candidate_long["Horizon"] == h)
        & (candidate_long["Year"] == year)
    ].copy()
    target = expected_target_frame(cur, h, year)
    expected_n = int(target["Forecast_Origin"].nunique())
    if g.empty or expected_n == 0:
        return pd.DataFrame(), [], expected_n

    coverage = (
        g.groupby("Model")["Forecast_Origin"].nunique()
        .div(expected_n)
    )
    eligible = sorted(coverage[coverage >= MIN_MODEL_COVERAGE].index.tolist())
    if not eligible:
        return pd.DataFrame(), [], expected_n

    wide = g[g["Model"].isin(eligible)].pivot_table(
        index="Forecast_Origin", columns="Model",
        values="Variance_Forecast", aggfunc="first"
    )
    # Use identical dates for every eligible candidate's inner-year score.
    common = wide.reindex(columns=eligible).dropna(subset=eligible)
    common_n = len(common)
    if common_n < max(30, int(np.ceil(MIN_COMMON_ORIGIN_COVERAGE * expected_n))):
        return pd.DataFrame(), [], expected_n

    common = common.reset_index().merge(target, on="Forecast_Origin", how="inner")
    rows = []
    for name in eligible:
        rows.append({
            "Model": name,
            "QLIKE": qlike(common["Realized_Variance"], common[name]),
            "N": len(common),
            "Coverage_Rate_Original": float(coverage[name]),
            "Common_Origins": common_n
        })
    rank = pd.DataFrame(rows).dropna(subset=["QLIKE"]).sort_values("QLIKE")
    return rank, common["Forecast_Origin"].tolist(), expected_n

def build_blend(cur, h, year, models, origins=None, ewma_fallback=False):
    """Build an equal-weight blend. Optional EWMA fallback is explicit and auditable."""
    target = expected_target_frame(cur, h, year)
    if origins is not None:
        origins = pd.to_datetime(pd.Series(origins))
        target = target[target["Forecast_Origin"].isin(set(origins))]
    if target.empty:
        return pd.DataFrame()

    g = candidate_long[
        (candidate_long["Currency"] == cur)
        & (candidate_long["Horizon"] == h)
        & (candidate_long["Year"] == year)
        & candidate_long["Model"].isin(models)
    ].copy()
    if g.empty:
        wide = pd.DataFrame(index=pd.DatetimeIndex([], name="Forecast_Origin"))
    else:
        wide = g.pivot_table(
            index="Forecast_Origin", columns="Model",
            values="Variance_Forecast", aggfunc="first"
        )
    # Retain every selected component column, even when a component has no forecasts
    # in this outer year. Its missing values will trigger the explicit EWMA fallback.
    wide = wide.reindex(columns=models)
    out = target.merge(wide.reset_index(), on="Forecast_Origin", how="left")
    complete = out[models].notna().all(axis=1)
    out["Blend_Forecast"] = out[models].mean(axis=1, skipna=False)
    out["Forecast_Fallback_EWMA"] = False

    if ewma_fallback:
        ewma = candidate_long[
            (candidate_long["Currency"] == cur)
            & (candidate_long["Horizon"] == h)
            & (candidate_long["Year"] == year)
            & (candidate_long["Model"] == "EWMA94")
        ][["Forecast_Origin", "Variance_Forecast"]].rename(
            columns={"Variance_Forecast": "EWMA94_Forecast"}
        )
        out = out.merge(ewma, on="Forecast_Origin", how="left")
        fallback_mask = ~complete | ~np.isfinite(out["Blend_Forecast"])
        out.loc[fallback_mask, "Blend_Forecast"] = out.loc[fallback_mask, "EWMA94_Forecast"]
        out["Forecast_Fallback_EWMA"] = fallback_mask
        out = out[np.isfinite(out["Blend_Forecast"]) & (out["Blend_Forecast"] > 0)]
    else:
        out = out.loc[complete].copy()
    return out

selection_rows = []
final_parts = []
selection_audit_rows = []

for cur in CURRENCIES:
    for h in HORIZONS:
        for year in OUTER_YEARS:
            inner = year - 1
            rank, common_origins, expected_n = model_year_table(cur, h, inner)
            if rank.empty:
                selection_audit_rows.append({
                    "Currency": cur, "Horizon": h, "Outer_Year": year,
                    "Inner_Year": inner, "Status": "NO_ELIGIBLE_COMMON_SAMPLE",
                    "Expected_Inner_Origins": expected_n, "Common_Inner_Origins": len(common_origins)
                })
                continue

            ranked = rank["Model"].tolist()
            strategies = {"TOP1": ranked[:1], "TOP2_EQ": ranked[:2], "TOP3_EQ": ranked[:3]}
            scores = []
            for strat, mods in strategies.items():
                b = build_blend(cur, h, inner, mods, origins=common_origins, ewma_fallback=False)
                if b.empty:
                    continue
                scores.append({
                    "Strategy": strat,
                    "Models": " + ".join(mods),
                    "Inner_QLIKE": qlike(b["Realized_Variance"], b["Blend_Forecast"]),
                    "Inner_Origins": len(b)
                })
            st = pd.DataFrame(scores).dropna().sort_values("Inner_QLIKE")
            if st.empty:
                selection_audit_rows.append({
                    "Currency": cur, "Horizon": h, "Outer_Year": year,
                    "Inner_Year": inner, "Status": "NO_VALID_STRATEGY",
                    "Expected_Inner_Origins": expected_n, "Common_Inner_Origins": len(common_origins)
                })
                continue

            chosen = st.iloc[0]
            mods = chosen["Models"].split(" + ")
            out = build_blend(cur, h, year, mods, origins=None, ewma_fallback=True)
            if out.empty or "EWMA94_Forecast" not in out.columns:
                selection_audit_rows.append({
                    "Currency": cur, "Horizon": h, "Outer_Year": year,
                    "Inner_Year": inner, "Status": "NO_OUTER_FORECAST",
                    "Expected_Inner_Origins": expected_n, "Common_Inner_Origins": len(common_origins)
                })
                continue

            out["Currency"] = cur
            out["Horizon"] = h
            out["Validation_Year"] = year
            out["Selected_Strategy"] = chosen["Strategy"]
            out["Selected_Models"] = chosen["Models"]
            final_parts.append(out)

            selection_rows.append({
                "Currency": cur, "Horizon": h, "Outer_Year": year, "Inner_Year": inner,
                "Selected_Strategy": chosen["Strategy"], "Selected_Models": chosen["Models"],
                "Inner_QLIKE": chosen["Inner_QLIKE"], "Inner_Origins": chosen["Inner_Origins"],
                "Eligible_Model_Count": len(rank),
                "Best_Inner_Single_Model": rank.iloc[0]["Model"],
                "Best_Inner_Single_QLIKE": rank.iloc[0]["QLIKE"],
                "Outer_Origins": len(out),
                "Outer_EWMA_Fallback_Count": int(out["Forecast_Fallback_EWMA"].sum()),
                "Outer_EWMA_Fallback_Rate": float(out["Forecast_Fallback_EWMA"].mean())
            })
            selection_audit_rows.append({
                "Currency": cur, "Horizon": h, "Outer_Year": year, "Inner_Year": inner,
                "Status": "SELECTED",
                "Expected_Inner_Origins": expected_n, "Common_Inner_Origins": len(common_origins)
            })

if not final_parts:
    raise RuntimeError("No final volatility forecasts were generated. Review candidate coverage and forecast audit.")

final_predictions = pd.concat(final_parts, ignore_index=True)
selections = pd.DataFrame(selection_rows)
selection_audit = pd.DataFrame(selection_audit_rows)
display(selections)
display(selection_audit)


In [ ]:
pooled=[]
for (cur,h),g in final_predictions.groupby(["Currency","Horizon"]):
    m=np.isfinite(g["Realized_Variance"])&np.isfinite(g["Blend_Forecast"])&np.isfinite(g["EWMA94_Forecast"])
    x=g.loc[m]
    fq=qlike(x["Realized_Variance"],x["Blend_Forecast"])
    eq=qlike(x["Realized_Variance"],x["EWMA94_Forecast"])
    fm=variance_mse(x["Realized_Variance"],x["Blend_Forecast"])
    em=variance_mse(x["Realized_Variance"],x["EWMA94_Forecast"])
    stat,p=dm_loss_test(x["Realized_Variance"],x["Blend_Forecast"],x["EWMA94_Forecast"],h)
    pooled.append({
        "Currency":cur,"Horizon":h,"Observations":len(x),
        "EWMA_QLIKE":eq,"Final_QLIKE":fq,"QLIKE_Gain_vs_EWMA":eq-fq,
        "EWMA_Variance_MSE":em,"Final_Variance_MSE":fm,
        "MSE_Skill_vs_EWMA":1-fm/em if em>0 else np.nan,
        "EWMA_Vol_RMSE":vol_rmse(x["Realized_Variance"],x["EWMA94_Forecast"]),
        "Final_Vol_RMSE":vol_rmse(x["Realized_Variance"],x["Blend_Forecast"]),
        "DM_Stat_QLIKE_vs_EWMA":stat,"DM_pvalue_QLIKE_vs_EWMA":p,
        "Beats_EWMA_QLIKE":fq<eq
    })
pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"])
display(pooled_results)
print("Tasks beating EWMA on primary QLIKE:",int(pooled_results["Beats_EWMA_QLIKE"].sum()),"/",len(pooled_results))

In [ ]:
yearly=[]
for (cur,h,year),g in final_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    fq=qlike(g["Realized_Variance"],g["Blend_Forecast"])
    eq=qlike(g["Realized_Variance"],g["EWMA94_Forecast"])
    stat,p=dm_loss_test(g["Realized_Variance"],g["Blend_Forecast"],g["EWMA94_Forecast"],h)
    yearly.append({
        "Currency":cur,"Horizon":h,"Validation_Year":year,"Observations":len(g),
        "EWMA_QLIKE":eq,"Final_QLIKE":fq,"QLIKE_Gain_vs_EWMA":eq-fq,
        "Final_Vol_RMSE":vol_rmse(g["Realized_Variance"],g["Blend_Forecast"]),
        "DM_Stat_QLIKE_vs_EWMA":stat,"DM_pvalue_QLIKE_vs_EWMA":p,
        "Beats_EWMA":fq<eq
    })
yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"])
display(yearly_results)

In [ ]:
# Secondary one-day 95% VaR exceedance diagnostic.
var_rows=[]
z95=stats.norm.ppf(0.05)
for cur in CURRENCIES:
    g=final_predictions[(final_predictions["Currency"]==cur)&(final_predictions["Horizon"]==1)].copy()
    tmp=series[cur][["Date","Return"]].copy(); tmp["Next_Return"]=tmp["Return"].shift(-1)
    g=g.merge(tmp[["Date","Next_Return"]].rename(columns={"Date":"Forecast_Origin"}),on="Forecast_Origin",how="left")
    sigma=np.sqrt(np.maximum(g["Blend_Forecast"].to_numpy(float),EPS))
    threshold=z95*sigma
    actual=g["Next_Return"].to_numpy(float)
    m=np.isfinite(actual)&np.isfinite(threshold)
    exc=actual[m]<threshold[m]
    var_rows.append({
        "Currency":cur,"Observations":int(m.sum()),"Expected_Exceedance_Rate":0.05,
        "Observed_Exceedance_Rate":float(exc.mean()) if m.sum() else np.nan,
        "Exceedances":int(exc.sum())
    })
var_diag=pd.DataFrame(var_rows)
display(var_diag)

In [ ]:
final_predictions.to_csv(PRED_DIR / "12_volatility_final_predictions_2019_2023.csv", index=False)
pooled_results.to_csv(TABLE_DIR / "12_volatility_pooled_results.csv", index=False)
yearly_results.to_csv(TABLE_DIR / "12_volatility_yearly_results.csv", index=False)
selections.to_csv(TABLE_DIR / "12_volatility_selections.csv", index=False)
var_diag.to_csv(TABLE_DIR / "12_volatility_var_diagnostic.csv", index=False)
selection_audit.to_csv(TABLE_DIR / "12_volatility_selection_audit.csv", index=False)

quality_gate = pd.DataFrame([
    ["Holdout isolation", "PASS", "2024–2025 never loaded"],
    ["Volatility target", "PASS", "Future realized variance from squared log returns"],
    ["Forecast origin", "PASS", "Return observed at t is used only for forecasts of t+1 onward"],
    ["EWMA timing", "PASS", "EWMA update includes the return observed at the forecast origin"],
    ["Fit stability", "AUDIT", "Non-converged/implausible fits are rejected and logged"],
    ["Extreme forecasts", "AUDIT", f"Raw forecast/EWMA ratios above {MAX_RAW_TO_EWMA_RATIO:g} are rejected and logged"],
    ["Model selection", "PASS", "Eligible models ranked on common origins in previous year only"],
    ["Missing component forecasts", "FALLBACK", "EWMA fallback used explicitly; counts saved in selection table"],
    ["Benchmark", "PASS", "EWMA volatility benchmark; no misuse of point-forecast RW"],
    ["Primary loss", "PASS", "QLIKE"],
    ["Asymmetry", "PASS", "EGARCH and GJR-GARCH candidates"],
    ["Heavy tails", "PASS", "Student-t candidates"]
], columns=["Area", "Status", "Evidence"])
display(quality_gate)
quality_gate.to_csv(TABLE_DIR / "12_volatility_quality_gate.csv", index=False)

print("\nSCRIPT 12 COMPLETE — CORRECTED VERSION")
print("Primary score:", int(pooled_results["Beats_EWMA_QLIKE"].sum()), "/", len(pooled_results),
      "tasks beat EWMA on pooled QLIKE")
print("Candidate predictions:", PRED_DIR / "12_volatility_candidate_predictions_2018_2023.csv")
print("Forecast audit:", TABLE_DIR / "12_volatility_forecast_audit.csv")
print("Candidate coverage:", TABLE_DIR / "12_volatility_candidate_coverage.csv")
print("Selection audit:", TABLE_DIR / "12_volatility_selection_audit.csv")
print("2024–2025 remains untouched")


# How to interpret the corrected results

The headline is **not** “beat Random Walk”, because Random Walk is a point-forecast benchmark. The correct volatility headline is:

> **X/12 currency–horizon tasks beat the EWMA volatility benchmark on pooled QLIKE.**

A lower pooled QLIKE is encouraging, but stronger claims require stability across years, adequate coverage, and supportive loss-differential evidence. Do not claim statistical superiority merely because pooled QLIKE is lower.

Review these audit outputs before accepting the results:
- `12_volatility_forecast_audit.csv`: rejected fits and forecasts, including numerical guardrail events.
- `12_volatility_candidate_coverage.csv`: available versus expected origins for each model.
- `12_volatility_selection_audit.csv`: inner-year eligibility and selection status.
- `12_volatility_selections.csv`: selected strategy, outer-year coverage, and EWMA fallback rate.

The notebook restricts all modelling and selection to development data before 2024. Keep the 2024–2025 final holdout untouched until the development methodology is frozen.
